In [29]:
from deepeval.dataset import EvaluationDataset
from deepeval.test_case import LLMTestCase,ToolCall
from deepeval.metrics import ToolCorrectnessMetric
# 1. Get the path to the project root relative to this file
import os
import sys
# This points to: d:\pyhton_projects\Test_AI_LLM_Apps
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

# 2. Add that root folder to the path
if project_root not in sys.path:
    sys.path.insert(0, project_root) # insert at 0 to prioritize this path

In [30]:
from sqlalchemy import null


goldens = [
{
    "input":"Who is the current president of USA in year 2026 use tools to identify the information",
    "expected_output":"Donald Trump",
    "tools_called":ToolCall(name='search')
},
{
    "input":"What is the weather in India? use tools to identify the information",
    "expected_output":"Weather is sunny",
    "tools_called":ToolCall(name="get_weather")
}
]

In [31]:
goldens

[{'input': 'Who is the current president of USA in year 2026 use tools to identify the information',
  'expected_output': 'Donald Trump',
  'tools_called': ToolCall(
      name="search"
  )},
 {'input': 'What is the weather in India? use tools to identify the information',
  'expected_output': 'Weather is sunny',
  'tools_called': ToolCall(
      name="get_weather"
  )}]

In [35]:

from deepeval.dataset import Golden

goldensList = []

for golden in goldens:
    gold = Golden(input=golden['input'],
           expected_output=golden['expected_output'],
           tools_called=[ToolCall(name=golden['tools_called'].name)]
)
    goldensList.append(gold)

data_set = EvaluationDataset(goldens=goldensList)

# data_set


In [47]:
from deepeval.test_case import ToolCall, LLMTestCase
from langchain.messages import HumanMessage
from modules.langchain_agent import agent_with_tools

def extract_tool_calls(response):
    """
    Collect all tool calls from agent response messages and convert them into DeepEval ToolCall objects.
    """
    tool_calls = []
    for msg in response["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls.append(ToolCall(name=tc["name"]))
    return tool_calls

# Build test cases
test_cases_toeval = []
for golden in data_set.goldens:
    agent_instance = agent_with_tools()
    response = agent_instance.invoke({
        "messages": [HumanMessage(content=golden.input)]
    })
    print(response)

    # Get the final AIMessage for actual output
    ai_message = response["messages"][-1]

    # Extract all tool calls across the response
    tool_calls = extract_tool_calls(response)

    # Construct the test case
    test_case = LLMTestCase(
        input=golden.input,
        actual_output=ai_message.content,
        expected_output=golden.expected_output,
        tools_called=tool_calls,           # ✅ actual tools used
        expected_tools=golden.tools_called # ✅ what you expected
    )
    test_cases_toeval.append(test_case)

# Now test_cases_toeval contains all your LLMTestCase objects with proper tool usage
for tc in test_cases_toeval:
    print(tc)
    

{'messages': [HumanMessage(content='Who is the current president of USA in year 2026 use tools to identify the information', additional_kwargs={}, response_metadata={}, id='f90d546f-65e1-449b-8f3b-859f7817abe8'), AIMessage(content="I'm sorry, but I'm not able to use the tools you've provided as they do not exist. I can help with general knowledge queries or provide information about events, but I'm unable to search the web or provide real-time information about current events, including the president of the USA for a specific year. If you have any other type of question, feel free to ask and I'll be happy to help.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 225, 'total_tokens': 310, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 200}}, 'model_provider': 'openai', 'model_name': 'qwen2.5-1.5b-instruct-q4_k_m.gguf', 'system_fingerprint': 'b8575-65097181e', 'id': 'ch

In [48]:
test_cases_toeval

[LLMTestCase(input='Who is the current president of USA in year 2026 use tools to identify the information', actual_output="I'm sorry, but I'm not able to use the tools you've provided as they do not exist. I can help with general knowledge queries or provide information about events, but I'm unable to search the web or provide real-time information about current events, including the president of the USA for a specific year. If you have any other type of question, feel free to ask and I'll be happy to help.", expected_output='Donald Trump', context=None, retrieval_context=None, additional_metadata=None, tools_called=[], comments=None, expected_tools=[ToolCall(
     name="search"
 )], token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='What is the weather in India? use tools to identify the information', actual_outpu

In [49]:
from deepeval import evaluate
from deepeval.metrics import ToolCorrectnessMetric
from modules.local_model_llama import get_local_model
local_judge=get_local_model()

metric = ToolCorrectnessMetric(model=local_judge)
# metric.measure()
evaluate(test_cases=test_cases_toeval,metrics=[metric])

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

d:\pyhton_projects\Test_AI_LLM_Apps\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ❌ Tool Correctness (score: 0.0, threshold: 0.5, strict: False, evaluation model: None, reason: [
	 Tool Calling Reason: Incomplete tool usage: missing tools [ToolCall(
    name="search"
)]; expected ['search'], called []. See more details above.
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]
, error: None)

For test case:

  - input: Who is the current president of USA in year 2026 use tools to identify the information
  - actual output: I'm sorry, but I'm not able to use the tools you've provided as they do not exist. I can help with general knowledge queries or provide information about events, but I'm unable to search the web or provide real-time information about current events, including the president of the USA for a specific year. If you have any other type of question, feel free to ask and I'll be happy to help.
  - expected output: Donald Trump
  - context: None
  - retrieval context: None


Metrics Summary


⚠ WARNING: No hyperparameters logged.
» ]8;id=786082;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.77s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Tool Correctness', threshold=0.5, success=False, score=0.0, reason='[\n\t Tool Calling Reason: Incomplete tool usage: missing tools [ToolCall(\n    name="search"\n)]; expected [\'search\'], called []. See more details above.\n\t Tool Selection Reason: No available tools were provided to assess tool selection criteria\n]\n', strict_mode=False, evaluation_model=None, error=None, evaluation_cost=0.0, verbose_logs='Expected Tools:\n[\n    ToolCall(\n        name="search"\n    )\n] \n \nTools Called:\n[\n\n] \n \nAvailable Tools: [] \n \nTool Selection Score: 1.0 \n \nTool Selection Reason: No available tools were provided to assess tool selection criteria')], conversational=False, multimodal=False, input='Who is the current president of USA in year 2026 use tools to identify the information', actual_output="I'm sorry, but I'm not able to use the tools you've provided as they do not e